# 05 Review Signal Extraction — Facial Skincare


In [1]:
from google.colab import drive
drive.mount("/content/drive")


Mounted at /content/drive


In [2]:
from pathlib import Path
from collections import Counter, defaultdict
import html
import hashlib
import json
import os
import platform
import re
import socket

import numpy as np
import pandas as pd
import pyarrow.parquet as pq
from IPython.display import display

pd.set_option("display.max_columns", 200)
pd.set_option("display.max_colwidth", 300)
pd.set_option("display.width", 200)


In [3]:
# =========================================================
# Configuration
# =========================================================
CATEGORY_ID = "face"
CATEGORY_FOLDER = "facial_skincare"
CATEGORY_LABEL = "Facial Skincare"
STAGE = "stage0_review_signal_extraction"
REVIEW_SIGNAL_SCHEMA_VERSION = "review_signal_v2_full_eligible_pool"

PROJECT_ROOT = Path("/content/drive/MyDrive/thesis_recsys/categories") / CATEGORY_FOLDER
os.chdir(PROJECT_ROOT)

ELIGIBLE_POOL_PATH = PROJECT_ROOT / "data" / "processed" / "user_sampling" / "face_final_sampling_pool.parquet"
USER_SAMPLE_PATH = PROJECT_ROOT / "data" / "processed" / "user_sampling" / "face_user_regime_sample.parquet"
USER_SAMPLE_MANIFEST_PATH = PROJECT_ROOT / "data" / "processed" / "user_sampling" / "face_user_regime_sampling_manifest.json"
RAW_REVIEWS_PATH = PROJECT_ROOT / "data" / "raw" / "reviews_Skin_Care_Face_W2_2019_2023.parquet"
ITEM_SCHEMA_PATH = PROJECT_ROOT / "data" / "processed" / "items" / "face_item_schema_full.parquet"

REVIEW_SIGNAL_DIR = PROJECT_ROOT / "data" / "processed" / "review_signals"
OUTPUT_DIR = PROJECT_ROOT / "outputs" / "stage0_review_signals"
REVIEW_SIGNAL_PATH = REVIEW_SIGNAL_DIR / "face_review_signals.parquet"
REVIEW_SIGNAL_CSV_PATH = REVIEW_SIGNAL_DIR / "face_review_signals.csv"
REVIEW_SIGNAL_MANIFEST_PATH = OUTPUT_DIR / "face_review_signal_manifest.json"

EXPECTED_TARGET_SELECTION_MODE = "recent_eligible_review_rank_le5"
MAX_TARGET_RANK_ALLOWED = 5
EXPECTED_EVALUATION_WINDOW_MONTHS = 9
EXPECTED_REGIMES = ["cold", "weak", "moderate", "strong"]
REGIME_ORDER = EXPECTED_REGIMES

EXPECTED_ELIGIBLE_REGIME_COUNTS = None
EXPECTED_ELIGIBLE_POOL_ROWS = None
EXPECTED_INITIAL_TARGET_PER_REGIME = None
EXPECTED_INITIAL_REGIME_COUNTS = None
EXPECTED_INITIAL_SAMPLE_ROWS = None
EXPECTED_REGIME_COUNTS = None
EXPECTED_SAMPLE_ROWS = None
EXPECTED_OUTPUT_ROWS = None
EXPECTED_TOTAL_N = None
EXPECTED_UNIQUE_USERS = None

SIGNAL_SOURCE = "heldout_target_review"
INTENDED_USE = "synthetic_query_generation_only"

HARMONIZED_FACET_POLICY_VERSION = "harmonized_v2_global_review_brand_retrieval_profile"
EXPECTED_BRAND_POLICY = "separate_preference_facet__query_unsafe__retrieval_profile_graph_safe"
EXPECTED_IDENTIFIER_POLICY = "diagnostic_only__excluded_from_query_safe_profile_and_canonical_retrieval_text"

HARMONIZED_REQUIRED_ITEM_SCHEMA_COLS = [
    "parent_asin",
    "title",
    "facet_brand_text",
    "facet_policy_version",
    "brand_policy",
    "identifier_policy",
]

QUERY_EVIDENCE_SOURCE = "target_review_only"
EXPECTED_MIN_TARGET_REVIEW_TOKENS = 8
EXPECTED_MIN_QUERY_SAFE_TOKEN_COUNT = 4
EXPECTED_MIN_QUERY_SAFE_SIGNAL_FAMILIES = 2
EXPECTED_MIN_QUERY_SAFE_SIGNAL_TOTAL = 2

REVIEW_SIGNAL_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

REQUIRED_INPUT_PATHS = {
    "eligible_pool": ELIGIBLE_POOL_PATH,
    "initial_sample": USER_SAMPLE_PATH,
    "user_sample_manifest": USER_SAMPLE_MANIFEST_PATH,
    "raw_reviews": RAW_REVIEWS_PATH,
    "item_schema": ITEM_SCHEMA_PATH,
}

missing_inputs = {name: str(path) for name, path in REQUIRED_INPUT_PATHS.items() if not path.exists()}
if missing_inputs:
    raise FileNotFoundError(f"Missing required input files: {missing_inputs}")

print("Input path:", ELIGIBLE_POOL_PATH)
print("Output path:", REVIEW_SIGNAL_PATH)


Input path: /content/drive/MyDrive/thesis_recsys/categories/facial_skincare/data/processed/user_sampling/face_final_sampling_pool.parquet
Output path: /content/drive/MyDrive/thesis_recsys/categories/facial_skincare/data/processed/review_signals/face_review_signals.parquet


### Common framework alignment note

This notebook performs deterministic, rule-based signal extraction over all 28,348 temporally eligible cases produced by Notebook 03. The held-out target review is the sole positive evidence source for synthetic-query construction. Target-item metadata are joined only to construct a negative dictionary for direct-cue removal and audit. They may remove prohibited title, brand, identifier, seller or manufacturer, product-line, and package cues, but they cannot add, repair, or score query evidence. Ratings, sentiment, prior user history, historical item-review signals, and LLM outputs are not used. Notebook 06 performs final query construction, query-sufficiency screening, and deterministic same-regime balancing to 572 Skincare cases per regime, or 2,288 cases in total.


In [4]:
# =========================================================
# Common Review Signal Framework Contract
# =========================================================
COMMON_REVIEW_SIGNAL_FRAMEWORK_VERSION = "common_review_signal_framework_v2_query_audit"

COMMON_REVIEW_SIGNAL_FAMILY_TO_ROLE = {'concern': 'need_benefit_concern', 'skin_type': 'target_context', 'benefit': 'need_benefit_concern', 'ingredient': 'ingredient_or_composition', 'product_type_or_form_texture': 'form_texture', 'usage_context': 'target_context'}

FACE_REVIEW_CATEGORY_ANCHORS = {"skin care", "skincare", "face", "facial", "skin care routine", "skincare routine"}
FACE_REVIEW_GENERIC_UTILITY_TOKENS = {
    "support", "supports", "help", "helps", "promote", "promotes", "boost",
    "formula", "blend", "complex", "product", "solution",
}
FACE_REVIEW_CONTEXT_DEPENDENT_TOKENS = {"daily", "natural", "wellness", "health", "care", "routine"}
FACE_REVIEW_SPECIFIC_SUPPORT_PATTERNS = {
    "skin barrier support": r"\bskin\s+barrier\s+support\b",
    "hydration support": r"\bhydration\s+support\b",
    "sensitive skin care": r"\bsensitive\s+skin\s+care\b",
    "acne care": r"\bacne\s+care\b",
}

COMMON_REVIEW_SIGNAL_CONTRACT = {
    "category_id": CATEGORY_ID if "CATEGORY_ID" in globals() else "face",
    "category_label": CATEGORY_LABEL,
    "framework_version": COMMON_REVIEW_SIGNAL_FRAMEWORK_VERSION,
    "family_to_common_role": COMMON_REVIEW_SIGNAL_FAMILY_TO_ROLE,
    "category_anchors": sorted(FACE_REVIEW_CATEGORY_ANCHORS),
    "generic_utility_tokens": sorted(FACE_REVIEW_GENERIC_UTILITY_TOKENS),
    "context_dependent_tokens": sorted(FACE_REVIEW_CONTEXT_DEPENDENT_TOKENS),
    "specific_support_patterns": FACE_REVIEW_SPECIFIC_SUPPORT_PATTERNS,
    "query_evidence_source": QUERY_EVIDENCE_SOURCE,
    "item_metadata_evidence_used": False,
    "historical_review_evidence_used": False,
    "user_prior_evidence_used": False,
    "query_safe_residual_column": "query_safe_residual_text",
    "policy_note": "Review-signal dictionaries and thresholds remain category-specific. Item metadata is used only to remove direct title, brand, and identifier shortcuts from the held-out target review.",
}

COMMON_REVIEW_SIGNAL_CONTRACT_PATH = OUTPUT_DIR / "face_review_signal_common_contract.json"
COMMON_REVIEW_SIGNAL_FAMILY_COVERAGE_PATH = OUTPUT_DIR / "face_review_signal_common_family_coverage.csv"


In [5]:
# =========================================================
# Text Cleaning and Query-Safe Helpers
# =========================================================
import html

def normalize_space(text):
    text = "" if text is None or pd.isna(text) else str(text)
    text = html.unescape(text)
    text = re.sub(r"<br\s*/?>", " ", text, flags=re.IGNORECASE)
    text = re.sub(r"<[^>]+>", " ", text)
    text = text.replace("\n", " ").replace("\t", " ")
    return re.sub(r"\s+", " ", text).strip()


def tokenize_text(text):
    return re.findall(r"[a-z0-9']+", normalize_space(text).lower())


def unique_keep_order(values):
    out = []
    seen = set()

    for value in values:
        value = normalize_space(value).lower()
        if value and value not in seen:
            seen.add(value)
            out.append(value)

    return out


def pipe_join(values):
    return " | ".join(unique_keep_order(values))


def split_pipe_values(value):
    value = normalize_space(value)
    if not value:
        return []

    return unique_keep_order(re.split(r"\s*\|\s*", value))


def json_safe_records(df):
    return json.loads(json.dumps(df.to_dict(orient="records"), default=str))


def load_json_if_exists(path):
    if not path.exists():
        return {}

    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


ASIN_PATTERN = re.compile(r"\bb0[a-z0-9]{8}\b", re.IGNORECASE)

PACKAGE_IDENTIFIER_PATTERN = re.compile(
    r"""
    \b(
        \d+(?:\.\d+)?[\s_\-/]*(?:oz|fl\.?[\s_\-/]*oz|ml|g|gram|grams|ct|count|pack|packs|pcs|piece|pieces|%|percent|
        mg|mcg|iu|spf)
        |spf[\s_\-/]*\d+
        |asin|seller|manufacturer|barcode|upc
    )\b
    """,
    re.IGNORECASE | re.VERBOSE,
)

SELLER_MANUFACTURER_PATTERN = re.compile(
    r"\b(sold by|seller|manufacturer|made by|distributed by|shipped by)\b",
    re.IGNORECASE,
)

DIRECT_CUE_METADATA_COLS = [
    "itemctx_facet_brand_text",
    "itemctx_brand_meta",
    "itemctx_brand_primary",
    "itemctx_brand_norm",
    "parent_asin",
]

TITLE_AUDIT_METADATA_COLS = [
    "itemctx_title",
]


def cue_tokens_from_metadata(row):
    cue_text_parts = []

    for col in DIRECT_CUE_METADATA_COLS:
        value = row.get(col, "")
        if pd.notna(value):
            cue_text_parts.extend(split_pipe_values(value))

    cue_tokens = set()

    for text in cue_text_parts:
        for token in tokenize_text(text):
            if len(token) >= 3:
                cue_tokens.add(token)

    return cue_tokens


def remove_exact_phrases(text, phrases):
    cleaned = normalize_space(text)

    for phrase in phrases:
        phrase = normalize_space(phrase)
        if len(phrase) < 3:
            continue

        phrase_tokens = re.findall(r"[a-z0-9]+", phrase.lower())
        if not phrase_tokens:
            continue

        if len(phrase_tokens) == 1:
            phrase_pattern = re.escape(phrase_tokens[0])
        else:
            phrase_pattern = r"[\s_\-–—]+".join(
                re.escape(token)
                for token in phrase_tokens
            )

        pattern = (
            r"(?<![a-z0-9])"
            + phrase_pattern
            + r"(?:['’]s)?"
            + r"(?![a-z0-9])"
        )

        cleaned = re.sub(pattern, " ", cleaned, flags=re.IGNORECASE)

    return normalize_space(cleaned)


def remove_direct_item_cues_with_audit(review_text, row):
    """
    Remove direct brand/identifier cues from held-out target review text.

    Returns an audit dict that separates:
    - actual direct cue removal
    - normalization/tokenization-only changes
    - package, title, brand, and identifier shortcuts removed before extraction
    """
    original_text = normalize_space(review_text)
    working_text = original_text

    audit = {
        "query_safe_review_text": "",
        "asin_pattern_removed": False,
        "seller_manufacturer_pattern_removed": False,
        "package_identifier_pattern_removed": False,
        "metadata_phrase_removed": False,
        "title_phrase_removed": False,
        "metadata_token_removed": False,
        "actual_direct_cue_removed": False,
        "normalization_changed": False,
    }

    after_asin = ASIN_PATTERN.sub(" ", working_text)
    audit["asin_pattern_removed"] = normalize_space(after_asin) != normalize_space(working_text)
    working_text = after_asin

    after_seller = SELLER_MANUFACTURER_PATTERN.sub(" ", working_text)
    audit["seller_manufacturer_pattern_removed"] = normalize_space(after_seller) != normalize_space(working_text)
    working_text = after_seller

    after_package_identifier = PACKAGE_IDENTIFIER_PATTERN.sub(" ", working_text)
    audit["package_identifier_pattern_removed"] = (
        normalize_space(after_package_identifier) != normalize_space(working_text)
    )
    working_text = after_package_identifier

    direct_phrases = []

    for col in DIRECT_CUE_METADATA_COLS:
        value = row.get(col, "")
        if pd.notna(value):
            direct_phrases.extend(split_pipe_values(value))

    before_phrase_removal = normalize_space(working_text)
    working_text = remove_exact_phrases(working_text, direct_phrases)
    audit["metadata_phrase_removed"] = normalize_space(working_text) != before_phrase_removal

    title_phrases = []
    for col in TITLE_AUDIT_METADATA_COLS:
        value = row.get(col, "")
        if pd.notna(value):
            title_phrases.extend(split_pipe_values(value))

    before_title_removal = normalize_space(working_text)
    working_text = remove_exact_phrases(working_text, title_phrases)
    audit["title_phrase_removed"] = (
        normalize_space(working_text) != before_title_removal
    )

    cue_tokens = cue_tokens_from_metadata(row)
    tokens_before = tokenize_text(working_text)

    safe_tokens = [
        token
        for token in tokens_before
        if token not in cue_tokens and token.rstrip("'s") not in cue_tokens
    ]

    audit["metadata_token_removed"] = len(safe_tokens) != len(tokens_before)

    final_text = " ".join(safe_tokens)

    # Tokenization can turn "$5/oz" into "5 oz" or "4-pack" into "4 pack".
    # Run the package/identifier scrub again on the normalized token text before audit.
    after_token_package_identifier = PACKAGE_IDENTIFIER_PATTERN.sub(" ", final_text)
    post_token_package_removed = normalize_space(after_token_package_identifier) != normalize_space(final_text)
    audit["package_identifier_pattern_removed"] = bool(
        audit["package_identifier_pattern_removed"] or post_token_package_removed
    )
    final_text = normalize_space(after_token_package_identifier)
    audit["query_safe_review_text"] = final_text

    audit["actual_direct_cue_removed"] = bool(
        audit["asin_pattern_removed"]
        or audit["seller_manufacturer_pattern_removed"]
        or audit["package_identifier_pattern_removed"]
        or audit["metadata_phrase_removed"]
        or audit["title_phrase_removed"]
        or audit["metadata_token_removed"]
    )

    audit["normalization_changed"] = (
        normalize_space(original_text).lower() != normalize_space(final_text).lower()
        and not audit["actual_direct_cue_removed"]
    )

    return audit


def remove_direct_item_cues(review_text, row):
    """
    Backward-compatible wrapper.

    Use remove_direct_item_cues_with_audit(...) in new cells when diagnostic counts
    are needed. This wrapper returns only the repaired query-safe text.
    """
    return remove_direct_item_cues_with_audit(review_text, row)["query_safe_review_text"]


def exact_phrase_present(text, phrase):
    phrase = normalize_space(phrase).lower()
    if len(phrase) < 4:
        return False

    text = normalize_space(text).lower()
    return bool(
        re.search(
            r"(?<![a-z0-9])" + re.escape(phrase) + r"(?![a-z0-9])",
            text,
        )
    )


def any_exact_phrase_present(text, row, cols):
    for col in cols:
        if col in row.index and exact_phrase_present(text, row.get(col, "")):
            return True

    return False


def forbidden_columns(columns, exact_cols, prefixes):
    forbidden = []

    for col in columns:
        col_str = str(col)
        if col_str in exact_cols or any(col_str.startswith(prefix) for prefix in prefixes):
            forbidden.append(col_str)

    return sorted(set(forbidden))


In [6]:
# =========================================================
# Load and Validate Notebook 03 Full Eligible Pool and Item Schema
# =========================================================
sample_manifest = load_json_if_exists(USER_SAMPLE_MANIFEST_PATH)
eligible_pool_df = pd.read_parquet(ELIGIBLE_POOL_PATH)
initial_sample_df = pd.read_parquet(USER_SAMPLE_PATH)
item_schema = pd.read_parquet(ITEM_SCHEMA_PATH)
raw_review_lookup_df = pd.read_parquet(
    RAW_REVIEWS_PATH,
    columns=["user_id", "parent_asin", "timestamp", "title", "text"],
)

OPTIONAL_LEGACY_ITEM_CONTEXT_COLS = ["brand_meta", "brand_primary", "brand_norm"]
REQUIRED_ITEM_SCHEMA_COLS = list(HARMONIZED_REQUIRED_ITEM_SCHEMA_COLS)

missing_item_schema_cols = sorted(set(REQUIRED_ITEM_SCHEMA_COLS) - set(item_schema.columns))
if missing_item_schema_cols:
    raise RuntimeError(
        "Item schema is missing harmonized Notebook 02 columns required by Notebook 05: "
        f"{missing_item_schema_cols}. Re-run 02_feature_engineering_face_revised_core_contract.ipynb first."
    )

item_schema["parent_asin"] = item_schema["parent_asin"].astype(str).str.strip()
if item_schema["parent_asin"].eq("").any() or item_schema["parent_asin"].isna().any():
    raise RuntimeError("Item schema parent_asin contains null or empty values.")
if item_schema["parent_asin"].duplicated().any():
    duplicate_preview = (
        item_schema.loc[item_schema["parent_asin"].duplicated(keep=False), ["parent_asin", "title"]]
        .sort_values("parent_asin")
        .head(20)
    )
    display(duplicate_preview)
    raise RuntimeError(
        "Item schema parent_asin must be unique. "
        f"Duplicated keys found: {int(item_schema['parent_asin'].duplicated().sum())}"
    )

policy_cols = ["facet_policy_version", "brand_policy", "identifier_policy"]
policy_missing_counts = {
    col: int(item_schema[col].fillna("").astype(str).str.strip().eq("").sum())
    for col in policy_cols
}
if any(count > 0 for count in policy_missing_counts.values()):
    raise RuntimeError(f"Item schema contains missing policy values: {policy_missing_counts}")

if set(item_schema["facet_policy_version"].dropna().astype(str).unique()) != {HARMONIZED_FACET_POLICY_VERSION}:
    raise RuntimeError("Unexpected facet_policy_version values in item schema.")
if set(item_schema["brand_policy"].dropna().astype(str).unique()) != {EXPECTED_BRAND_POLICY}:
    raise RuntimeError("Unexpected brand_policy values in item schema.")
if set(item_schema["identifier_policy"].dropna().astype(str).unique()) != {EXPECTED_IDENTIFIER_POLICY}:
    raise RuntimeError("Unexpected identifier_policy values in item schema.")

FORBIDDEN_ITEM_REVIEW_EXACT_COLS = {
    "concern_review_raw",
    "concern_review_raw_text",
    "concern_review_evidence_text",
    "has_review_evidence",
    "review_text",
    "review_text_agg_train",
    "target_review_text",
    "heldout_review_text",
    "review_body",
    "raw_review_text",
    "rating",
    "sentiment",
    "helpful_vote",
    "total_vote",
    "prompt",
    "response",
    "llm_response",
    "query_evidence",
}
FORBIDDEN_ITEM_REVIEW_PREFIXES = (
    "review_evidence_",
    "review_text_",
    "target_review_",
    "heldout_review_",
    "raw_review_",
)
forbidden_item_cols = forbidden_columns(
    item_schema.columns,
    FORBIDDEN_ITEM_REVIEW_EXACT_COLS,
    FORBIDDEN_ITEM_REVIEW_PREFIXES,
)
if forbidden_item_cols:
    raise RuntimeError(f"Forbidden raw/target-review item columns found in item schema: {forbidden_item_cols}")

required_manifest_keys = {
    "evaluation_window_months",
    "target_selection_mode",
    "max_target_rank_allowed",
    "target_rank_convention",
    "target_review_text_intended_use",
    "regimes",
    "query_convertibility_thresholds",
    "eligible_pool_rows",
    "eligible_pool_regime_counts",
    "final_balanced_sample_rows",
    "final_balanced_sample_regime_counts",
}
missing_manifest_keys = sorted(required_manifest_keys - set(sample_manifest.keys()))
if missing_manifest_keys:
    raise RuntimeError(f"Notebook 03 manifest is missing required keys: {missing_manifest_keys}.")
if int(sample_manifest["evaluation_window_months"]) != EXPECTED_EVALUATION_WINDOW_MONTHS:
    raise RuntimeError("Notebook 03 manifest evaluation_window_months mismatch.")
if sample_manifest["target_selection_mode"] != EXPECTED_TARGET_SELECTION_MODE:
    raise RuntimeError("Notebook 03 manifest target_selection_mode mismatch.")
if int(sample_manifest["max_target_rank_allowed"]) != MAX_TARGET_RANK_ALLOWED:
    raise RuntimeError("Notebook 03 manifest max_target_rank_allowed mismatch.")
if sample_manifest["target_review_text_intended_use"] != "synthetic_query_generation_only":
    raise RuntimeError("Notebook 03 manifest target_review_text_intended_use mismatch.")
if set(sample_manifest.get("regimes", [])) != set(EXPECTED_REGIMES):
    raise RuntimeError("Notebook 03 manifest regimes mismatch.")

query_thresholds = sample_manifest["query_convertibility_thresholds"]
expected_query_thresholds = {
    "MIN_TARGET_REVIEW_TOKENS": EXPECTED_MIN_TARGET_REVIEW_TOKENS,
    "MIN_QUERY_SAFE_TOKEN_COUNT": EXPECTED_MIN_QUERY_SAFE_TOKEN_COUNT,
    "MIN_QUERY_SAFE_SIGNAL_FAMILIES": EXPECTED_MIN_QUERY_SAFE_SIGNAL_FAMILIES,
    "MIN_QUERY_SAFE_SIGNAL_TOTAL": EXPECTED_MIN_QUERY_SAFE_SIGNAL_TOTAL,
}
observed_query_thresholds = {key: int(query_thresholds.get(key, -1)) for key in expected_query_thresholds}
if observed_query_thresholds != expected_query_thresholds:
    raise RuntimeError(
        "Notebook 03 query-evidence thresholds changed unexpectedly: "
        f"actual {observed_query_thresholds}, expected {expected_query_thresholds}."
    )

required_eligible_cols = {
    "case_id",
    "user_id",
    "parent_asin",
    "review_timestamp_ms",
    "review_datetime",
    "regime",
    "sampling_bracket",
    "target_selection_mode",
    "target_review_token_count",
    "query_safe_token_count",
    "query_safe_signal_family_count",
    "query_safe_signal_total_count",
    "query_convertible_flag",
    "query_convertibility_failure_reason",
}
missing_eligible_cols = sorted(required_eligible_cols - set(eligible_pool_df.columns))
if missing_eligible_cols:
    raise RuntimeError(f"Notebook 03 eligible pool is missing required columns for Notebook 05: {missing_eligible_cols}")

required_initial_cols = {"case_id", "user_id", "parent_asin", "regime", "sampling_bracket", "target_selection_mode", "target_rank_desc"}
missing_initial_cols = sorted(required_initial_cols - set(initial_sample_df.columns))
if missing_initial_cols:
    raise RuntimeError(f"Notebook 03 initial sample is missing required audit columns: {missing_initial_cols}")

for frame_name, frame in {"eligible pool": eligible_pool_df, "initial sample": initial_sample_df}.items():
    frame["case_id"] = frame["case_id"].astype(str).str.strip()
    frame["user_id"] = frame["user_id"].astype(str).str.strip()
    frame["parent_asin"] = frame["parent_asin"].astype(str).str.strip()
    for col in ["case_id", "user_id", "parent_asin"]:
        if frame[col].eq("").any() or frame[col].isna().any():
            raise RuntimeError(f"{frame_name} contains null or empty {col} values.")
    if frame["case_id"].duplicated().any():
        raise RuntimeError(f"{frame_name} must contain unique case_id values.")

if eligible_pool_df["user_id"].duplicated().any():
    raise RuntimeError("Notebook 03 full eligible pool must contain at most one eligible target per user_id.")
if initial_sample_df["user_id"].duplicated().any():
    raise RuntimeError("Notebook 03 initial sample must contain one row per user_id.")

eligible_pool_df["review_timestamp_ms"] = pd.to_numeric(eligible_pool_df["review_timestamp_ms"], errors="coerce")
if eligible_pool_df["review_timestamp_ms"].isna().any():
    raise RuntimeError("Notebook 03 full eligible pool contains missing target timestamps.")
eligible_pool_df["target_timestamp_ms"] = eligible_pool_df["review_timestamp_ms"].astype("int64")
eligible_pool_df["review_timestamp_ms"] = eligible_pool_df["target_timestamp_ms"]

if eligible_pool_df["review_datetime"].fillna("").astype(str).str.strip().eq("").any():
    raise RuntimeError("Notebook 03 full eligible pool contains missing target review datetimes.")
if set(eligible_pool_df["regime"].dropna().astype(str).unique()) != set(EXPECTED_REGIMES):
    raise RuntimeError("Notebook 03 full eligible pool contains unsupported regime values.")
if not eligible_pool_df["regime"].isin(EXPECTED_REGIMES).all():
    raise RuntimeError("Notebook 03 full eligible pool contains null or unsupported regime values.")
if not eligible_pool_df["target_selection_mode"].eq(EXPECTED_TARGET_SELECTION_MODE).all():
    raise RuntimeError(
        f"Notebook 05 expects target_selection_mode == {EXPECTED_TARGET_SELECTION_MODE} for every eligible row."
    )
if not initial_sample_df["target_selection_mode"].eq(EXPECTED_TARGET_SELECTION_MODE).all():
    raise RuntimeError(
        f"Notebook 05 expects target_selection_mode == {EXPECTED_TARGET_SELECTION_MODE} for every sampled row."
    )
if not eligible_pool_df["target_rank_desc"].between(1, MAX_TARGET_RANK_ALLOWED).all():
    raise RuntimeError(f"Eligible target_rank_desc must be between 1 and {MAX_TARGET_RANK_ALLOWED}.")
if not initial_sample_df["target_rank_desc"].between(1, MAX_TARGET_RANK_ALLOWED).all():
    raise RuntimeError(f"Sampled target_rank_desc must be between 1 and {MAX_TARGET_RANK_ALLOWED}.")
if not eligible_pool_df["query_convertible_flag"].astype(bool).all():
    raise RuntimeError("Every Notebook 03 eligible-pool row must be query-convertible.")
if eligible_pool_df["target_review_token_count"].lt(EXPECTED_MIN_TARGET_REVIEW_TOKENS).any():
    raise RuntimeError("Notebook 03 target-review token threshold mismatch.")
if eligible_pool_df["query_safe_token_count"].lt(EXPECTED_MIN_QUERY_SAFE_TOKEN_COUNT).any():
    raise RuntimeError("Notebook 03 query-safe token threshold mismatch.")
if eligible_pool_df["query_safe_signal_family_count"].lt(EXPECTED_MIN_QUERY_SAFE_SIGNAL_FAMILIES).any():
    raise RuntimeError("Notebook 03 signal-family threshold mismatch.")
if eligible_pool_df["query_safe_signal_total_count"].lt(EXPECTED_MIN_QUERY_SAFE_SIGNAL_TOTAL).any():
    raise RuntimeError("Notebook 03 signal-total threshold mismatch.")

eligible_regime_counts = (
    eligible_pool_df["regime"].value_counts().reindex(EXPECTED_REGIMES, fill_value=0).astype(int).to_dict()
)
manifest_eligible_regime_counts = {
    regime: int(sample_manifest["eligible_pool_regime_counts"].get(regime, 0))
    for regime in EXPECTED_REGIMES
}
if len(eligible_pool_df) != int(sample_manifest["eligible_pool_rows"]):
    raise RuntimeError("Eligible pool row count differs from Notebook 03 manifest.")
if eligible_regime_counts != manifest_eligible_regime_counts:
    raise RuntimeError(
        f"Eligible-pool regime counts differ from Notebook 03 manifest: "
        f"actual {eligible_regime_counts}, manifest {manifest_eligible_regime_counts}."
    )
EXPECTED_ELIGIBLE_REGIME_COUNTS = dict(eligible_regime_counts)
EXPECTED_ELIGIBLE_POOL_ROWS = int(len(eligible_pool_df))
EXPECTED_REGIME_COUNTS = dict(eligible_regime_counts)
EXPECTED_SAMPLE_ROWS = EXPECTED_ELIGIBLE_POOL_ROWS
EXPECTED_OUTPUT_ROWS = EXPECTED_ELIGIBLE_POOL_ROWS
EXPECTED_TOTAL_N = EXPECTED_OUTPUT_ROWS
EXPECTED_UNIQUE_USERS = EXPECTED_OUTPUT_ROWS

initial_regime_counts = (
    initial_sample_df["regime"].value_counts().reindex(EXPECTED_REGIMES, fill_value=0).astype(int).to_dict()
)
manifest_initial_regime_counts = {
    regime: int(sample_manifest["final_balanced_sample_regime_counts"].get(regime, 0))
    for regime in EXPECTED_REGIMES
}
if len(initial_sample_df) != int(sample_manifest["final_balanced_sample_rows"]):
    raise RuntimeError("Initial sample row count differs from Notebook 03 manifest.")
if initial_regime_counts != manifest_initial_regime_counts:
    raise RuntimeError("Initial-sample regime counts differ from Notebook 03 manifest.")
EXPECTED_INITIAL_REGIME_COUNTS = dict(initial_regime_counts)
if len(set(EXPECTED_INITIAL_REGIME_COUNTS.values())) != 1:
    raise RuntimeError(f"Initial-sample regime counts must be balanced: {EXPECTED_INITIAL_REGIME_COUNTS}")
EXPECTED_INITIAL_TARGET_PER_REGIME = int(next(iter(EXPECTED_INITIAL_REGIME_COUNTS.values())))
EXPECTED_INITIAL_SAMPLE_ROWS = int(len(initial_sample_df))

initial_case_ids = set(initial_sample_df["case_id"].astype(str))
eligible_case_ids = set(eligible_pool_df["case_id"].astype(str))
if not initial_case_ids.issubset(eligible_case_ids):
    raise RuntimeError("Initial sample contains case_id values outside the full eligible pool.")
eligible_pool_df["initial_selected"] = eligible_pool_df["case_id"].isin(initial_case_ids)
if int(eligible_pool_df["initial_selected"].sum()) != EXPECTED_INITIAL_SAMPLE_ROWS:
    raise RuntimeError("initial_selected membership count does not match the Notebook 03 initial sample size.")

raw_review_lookup_df["user_id"] = raw_review_lookup_df["user_id"].astype(str).str.strip()
raw_review_lookup_df["parent_asin"] = raw_review_lookup_df["parent_asin"].astype(str).str.strip()
raw_review_lookup_df["review_timestamp_ms"] = pd.to_numeric(raw_review_lookup_df["timestamp"], errors="coerce")
raw_review_lookup_df = raw_review_lookup_df.drop(columns=["timestamp"])
raw_review_lookup_df["target_review_text"] = (
    raw_review_lookup_df["title"].fillna("").astype(str).map(normalize_space)
    + " "
    + raw_review_lookup_df["text"].fillna("").astype(str).map(normalize_space)
).map(normalize_space)
raw_review_lookup_df = raw_review_lookup_df.drop(columns=["title", "text"])
raw_review_lookup_df = raw_review_lookup_df.dropna(subset=["review_timestamp_ms"])
raw_review_lookup_df["review_timestamp_ms"] = raw_review_lookup_df["review_timestamp_ms"].astype("int64")

RAW_REVIEW_JOIN_KEYS = ["user_id", "parent_asin", "review_timestamp_ms"]
pool_key_df = eligible_pool_df[RAW_REVIEW_JOIN_KEYS].drop_duplicates()
duplicate_raw_rows = raw_review_lookup_df.loc[
    raw_review_lookup_df.duplicated(RAW_REVIEW_JOIN_KEYS, keep=False)
].copy()
if len(duplicate_raw_rows):
    duplicate_pool_raw_rows = duplicate_raw_rows.merge(pool_key_df, on=RAW_REVIEW_JOIN_KEYS, how="inner")
    if len(duplicate_pool_raw_rows):
        ambiguous_duplicate_keys = (
            duplicate_pool_raw_rows.groupby(RAW_REVIEW_JOIN_KEYS)["target_review_text"]
            .nunique(dropna=False)
            .reset_index(name="target_review_text_versions")
        )
        ambiguous_duplicate_keys = ambiguous_duplicate_keys.loc[
            ambiguous_duplicate_keys["target_review_text_versions"].gt(1)
        ]
        if len(ambiguous_duplicate_keys):
            display(ambiguous_duplicate_keys.head(20))
            raise RuntimeError("Raw review lookup has ambiguous duplicate target-review rows for eligible cases.")

raw_review_lookup_df = raw_review_lookup_df.sort_values(RAW_REVIEW_JOIN_KEYS).drop_duplicates(
    RAW_REVIEW_JOIN_KEYS,
    keep="first",
)
eligible_cases_df = eligible_pool_df.merge(
    raw_review_lookup_df[RAW_REVIEW_JOIN_KEYS + ["target_review_text"]],
    on=RAW_REVIEW_JOIN_KEYS,
    how="left",
    indicator="target_review_merge_status",
    validate="one_to_one",
)
if eligible_cases_df["case_id"].duplicated().any():
    raise RuntimeError("Duplicate case_id rows were created during target-review merge.")
missing_target_review_rows = int(
    eligible_cases_df["target_review_text"].fillna("").astype(str).map(normalize_space).eq("").sum()
)
missing_target_review_join_rows = int(eligible_cases_df["target_review_merge_status"].ne("both").sum())
if missing_target_review_join_rows:
    missing_preview = eligible_cases_df.loc[
        eligible_cases_df["target_review_merge_status"].ne("both"),
        ["case_id", "user_id", "parent_asin", "review_timestamp_ms", "target_review_merge_status"],
    ].head(20)
    display(missing_preview)
    raise RuntimeError(
        "Full eligible pool contains target-review rows that cannot be joined to raw review text. "
        f"Missing joins: {missing_target_review_join_rows}."
    )

eligible_input_row_count = int(len(eligible_cases_df))
initial_selected_row_count = int(eligible_cases_df["initial_selected"].sum())
reserve_row_count = int(eligible_input_row_count - initial_selected_row_count)


In [7]:
# =========================================================
# Merge Item Metadata Context for Direct Cue Removal and Audit
# =========================================================
review_reputation_cols_present = [
    col
    for col in item_schema.columns
    if str(col).startswith("review_reputation_")
    or str(col).startswith("historical_review_")
    or str(col) == "review_reputation_facet_text"
]

ITEM_CONTEXT_COLS = [
    "parent_asin",
    "title",
    "facet_brand_text",
    "facet_policy_version",
    "brand_policy",
    "identifier_policy",
]

OPTIONAL_ITEM_CONTEXT_COLS = [
    "brand_meta",
    "brand_primary",
    "brand_norm",
]

ITEM_CONTEXT_COLS = ITEM_CONTEXT_COLS + [
    col
    for col in OPTIONAL_ITEM_CONTEXT_COLS
    if col in item_schema.columns and col not in ITEM_CONTEXT_COLS
]

missing_item_context_cols = sorted(set(HARMONIZED_REQUIRED_ITEM_SCHEMA_COLS) - set(item_schema.columns))
if missing_item_context_cols:
    raise RuntimeError(
        f"Item schema is missing required harmonized context columns: {missing_item_context_cols}"
    )

item_lookup = item_schema[ITEM_CONTEXT_COLS].copy()
item_lookup["parent_asin"] = item_lookup["parent_asin"].astype(str).str.strip()

if item_lookup["parent_asin"].eq("").any():
    raise RuntimeError("item_lookup parent_asin contains empty values.")

if item_lookup["parent_asin"].isna().any():
    raise RuntimeError("item_lookup parent_asin contains null values.")

if item_lookup["parent_asin"].duplicated().any():
    duplicate_preview = (
        item_lookup.loc[
            item_lookup["parent_asin"].duplicated(keep=False),
            ["parent_asin", "title"],
        ]
        .sort_values("parent_asin")
        .head(20)
    )

    display(duplicate_preview)

    raise RuntimeError(
        "item_lookup parent_asin must be unique before metadata merge. "
        f"Duplicated keys found: {int(item_lookup['parent_asin'].duplicated().sum())}"
    )

item_lookup = item_lookup.rename(
    columns={
        col: f"itemctx_{col}"
        for col in item_lookup.columns
        if col != "parent_asin"
    }
)

source = eligible_cases_df.copy()
source["parent_asin"] = source["parent_asin"].astype(str).str.strip()

if source["parent_asin"].eq("").any():
    raise RuntimeError("Eligible source parent_asin contains empty values.")

if source["parent_asin"].isna().any():
    raise RuntimeError("Eligible source parent_asin contains null values.")

source["target_review_text_for_extraction"] = (
    source["target_review_text"]
    .fillna("")
    .astype(str)
    .map(normalize_space)
)

source = source.merge(
    item_lookup,
    on="parent_asin",
    how="left",
    indicator="item_metadata_merge_status",
    validate="many_to_one",
)

source["item_metadata_exists"] = source["item_metadata_merge_status"].eq("both")

metadata_matched_rows = int(source["item_metadata_exists"].sum())
metadata_match_rate = metadata_matched_rows / len(source)

itemctx_title_non_empty = int(
    source["itemctx_title"].fillna("").astype(str).str.strip().ne("").sum()
)

itemctx_facet_brand_text_non_empty = int(
    source["itemctx_facet_brand_text"].fillna("").astype(str).str.strip().ne("").sum()
)

optional_itemctx_non_empty_counts = {}
for col in OPTIONAL_ITEM_CONTEXT_COLS:
    itemctx_col = f"itemctx_{col}"
    if itemctx_col in source.columns:
        optional_itemctx_non_empty_counts[itemctx_col] = int(
            source[itemctx_col].fillna("").astype(str).str.strip().ne("").sum()
        )


if metadata_matched_rows != EXPECTED_OUTPUT_ROWS:
    unmatched = source.loc[
        ~source["item_metadata_exists"],
        ["case_id", "user_id", "parent_asin", "item_metadata_merge_status"],
    ].head(20)

    display(unmatched)

    raise RuntimeError(
        f"Item metadata merge matched {metadata_matched_rows}/{EXPECTED_OUTPUT_ROWS} rows."
    )

if itemctx_title_non_empty != EXPECTED_OUTPUT_ROWS:
    raise RuntimeError(
        f"itemctx_title should be non-empty for all eligible rows. "
        f"Found {itemctx_title_non_empty}/{EXPECTED_OUTPUT_ROWS}."
    )

for policy_col, expected_value in {
    "itemctx_facet_policy_version": HARMONIZED_FACET_POLICY_VERSION,
    "itemctx_brand_policy": EXPECTED_BRAND_POLICY,
    "itemctx_identifier_policy": EXPECTED_IDENTIFIER_POLICY,
}.items():
    values = set(source[policy_col].dropna().astype(str).unique().tolist())

    if values != {expected_value}:
        raise RuntimeError(f"{policy_col} mismatch after merge: {sorted(values)}")

source["query_safe_review_text_before_direct_cue_repair"] = (
    source["target_review_text_for_extraction"].map(normalize_space)
)

cue_repair_audits = source.apply(
    lambda row: remove_direct_item_cues_with_audit(
        row["query_safe_review_text_before_direct_cue_repair"],
        row,
    ),
    axis=1,
)

cue_repair_audit_df = pd.DataFrame(cue_repair_audits.tolist(), index=source.index)

for col in cue_repair_audit_df.columns:
    source[col] = cue_repair_audit_df[col]

source["direct_cue_repair_applied"] = source["actual_direct_cue_removed"].astype(bool)

source["target_review_token_count_05"] = source["target_review_text_for_extraction"].map(
    lambda text: len(tokenize_text(text))
)

source["query_safe_token_count_05"] = source["query_safe_review_text"].map(
    lambda text: len(tokenize_text(text))
)

direct_cue_audit_rows = []

brand_audit_cols = [
    "itemctx_facet_brand_text",
    "itemctx_brand_meta",
    "itemctx_brand_primary",
    "itemctx_brand_norm",
]

identifier_audit_cols = [
    "parent_asin",
]

for _, row in source.iterrows():
    text = normalize_space(row.get("query_safe_review_text", ""))

    direct_cue_audit_rows.append({
        "case_id": row.get("case_id", ""),
        "user_id": row.get("user_id", ""),
        "target_parent_asin": row.get("parent_asin", ""),
        "contains_asin_pattern": bool(ASIN_PATTERN.search(text)),
        "contains_package_or_identifier_pattern": bool(PACKAGE_IDENTIFIER_PATTERN.search(text)),
        "contains_seller_manufacturer_pattern": bool(SELLER_MANUFACTURER_PATTERN.search(text)),
        "contains_exact_brand_phrase": any_exact_phrase_present(text, row, brand_audit_cols),
        "contains_exact_identifier_phrase": any_exact_phrase_present(text, row, identifier_audit_cols),
        "contains_exact_title_phrase_warning": any_exact_phrase_present(text, row, TITLE_AUDIT_METADATA_COLS),
    })

direct_cue_leakage_df = pd.DataFrame(direct_cue_audit_rows)

blocking_direct_cue_cols = [
    "contains_asin_pattern",
    "contains_package_or_identifier_pattern",
    "contains_seller_manufacturer_pattern",
    "contains_exact_brand_phrase",
    "contains_exact_identifier_phrase",
    "contains_exact_title_phrase_warning",
]

direct_cue_leakage_df["direct_cue_warning_count"] = (
    direct_cue_leakage_df[blocking_direct_cue_cols]
    .fillna(False)
    .sum(axis=1)
    .astype(int)
)
source["direct_cue_warning_count"] = direct_cue_leakage_df["direct_cue_warning_count"].to_numpy()

direct_leakage_flag_count = int(
    direct_cue_leakage_df[blocking_direct_cue_cols]
    .fillna(False)
    .any(axis=1)
    .sum()
)

package_or_identifier_warning_count = int(
    direct_cue_leakage_df["contains_package_or_identifier_pattern"]
    .fillna(False)
    .sum()
)

title_exact_leak_warning_count = int(
    direct_cue_leakage_df["contains_exact_title_phrase_warning"]
    .fillna(False)
    .sum()
)

actual_direct_cue_removed_rows = int(source["actual_direct_cue_removed"].fillna(False).sum())
normalization_only_changed_rows = int(source["normalization_changed"].fillna(False).sum())
asin_removed_rows = int(source["asin_pattern_removed"].fillna(False).sum())
seller_manufacturer_removed_rows = int(source["seller_manufacturer_pattern_removed"].fillna(False).sum())
package_identifier_removed_rows = int(
    source["package_identifier_pattern_removed"].fillna(False).sum()
)
metadata_phrase_removed_rows = int(source["metadata_phrase_removed"].fillna(False).sum())
title_phrase_removed_rows = int(source["title_phrase_removed"].fillna(False).sum())
metadata_token_removed_rows = int(source["metadata_token_removed"].fillna(False).sum())

if direct_leakage_flag_count > 0:
    flagged_preview = direct_cue_leakage_df.loc[
        direct_cue_leakage_df[blocking_direct_cue_cols].fillna(False).any(axis=1),
        ["case_id", "user_id", "target_parent_asin"] + blocking_direct_cue_cols,
    ].head(20)

    display(flagged_preview)

    raise RuntimeError(
        "Direct brand/title/identifier/package shortcut leakage remains after direct-cue repair. "
        f"Flagged rows: {direct_leakage_flag_count}."
    )


In [8]:
# =========================================================
# Deterministic Query-Safe Signal Dictionaries
# =========================================================
SIGNAL_PATTERNS = {
    "concern": {
        "acne": [r"\bacne\b", r"\bbreakouts?\b", r"\bblemishes?\b", r"\bpimples?\b"],
        "redness": [r"\bredness\b", r"\bred skin\b"],
        "dark spots": [r"\bdark spots?\b", r"\bhyperpigmentation\b", r"\buneven tone\b"],
        "pores": [r"\bpores?\b", r"\blarge pores?\b"],
        "dryness": [r"\bdry(ness)?\b", r"\bdehydrat(ed|ion)\b", r"\bflaky\b"],
        "oiliness": [r"\boily\b", r"\bgreasy\b", r"\bshine\b"],
        "dullness": [r"\bdull(ness)?\b"],
        "fine lines": [r"\bfine lines?\b", r"\bwrinkles?\b"],
        "texture": [r"\btexture\b", r"\brough\b", r"\bbumpy\b"],
        "sensitivity": [r"\bsensitiv(e|ity)\b"],
    },
    "skin_type": {
        "dry skin": [r"\bdry skin\b"],
        "oily skin": [r"\boily skin\b"],
        "combination skin": [r"\bcombination skin\b", r"\bcombo skin\b"],
        "sensitive skin": [r"\bsensitive skin\b", r"\bsensitiv(e|ity)\b"],
        "acne-prone skin": [r"\bacne prone\b", r"\bacne-prone\b"],
    },
    "benefit": {
        "hydrating": [r"\bhydrat\w*", r"\bmoisturi[sz]\w*"],
        "soothing": [r"\bsooth\w*", r"\bcalm\w*"],
        "brightening": [r"\bbrighten\w*", r"\bglow\w*"],
        "smoothing": [r"\bsmooth\w*", r"\bsoften\w*"],
        "firming": [r"\bfirm\w*", r"\btighten\w*"],
        "cleansing": [r"\bcleanse\w*", r"\bremoves? makeup\b"],
        "exfoliating": [r"\bexfoliat\w*", r"\bpeel\b"],
    },
    "ingredient": {
        "hyaluronic acid": [r"\bhyaluronic acid\b"],
        "niacinamide": [r"\bniacinamide\b"],
        "retinol": [r"\bretinol\b", r"\bretinoid\b"],
        "vitamin c": [r"\bvitamin c\b"],
        "ceramide": [r"\bceramides?\b"],
        "salicylic acid": [r"\bsalicylic acid\b", r"\bbha\b"],
        "glycolic acid": [r"\bglycolic acid\b", r"\baha\b"],
        "centella": [r"\bcentella\b", r"\bcica\b"],
        "aloe": [r"\baloe\b"],
        "tea tree": [r"\btea tree\b"],
    },
    "product_type_or_form_texture": {
        "cleanser": [r"\bcleanser\b", r"\bface wash\b"],
        "serum": [r"\bserum\b"],
        "moisturizer": [r"\bmoisturi[sz]er\b", r"\bcream\b", r"\blotion\b"],
        "toner": [r"\btoner\b"],
        "mask": [r"\bmask\b"],
        "sunscreen": [r"\bsunscreen\b"],
        "balm": [r"\bbalm\b"],
        "oil": [r"\boil\b"],
        "gel": [r"\bgel\b"],
        "foam": [r"\bfoam\w*\b"],
        "lightweight": [r"\blight ?weight\b"],
        "non-greasy": [r"\bnon greasy\b", r"\bnon-greasy\b"],
    },
    "usage_context": {
        "morning routine": [r"\bmorning\b", r"\bam routine\b"],
        "night routine": [r"\bnight\b", r"\bnighttime\b", r"\bpm routine\b", r"\bbefore bed\b"],
        "daily use": [r"\bdaily\b", r"\bevery day\b", r"\btwice a day\b"],
        "under makeup": [r"\bunder makeup\b", r"\blayers? well\b"],
        "seasonal context": [r"\bwinter\b", r"\bsummer\b", r"\bdry weather\b", r"\bhumid\b"],
    },
}

QUERY_SAFE_SIGNAL_FAMILIES = list(SIGNAL_PATTERNS.keys())
SENTIMENT_LIKE_FAMILIES_REMOVED = ["positive_preference", "negative_preference"]


In [9]:
# =========================================================
# Extract Deterministic Held-Out Target-Review Signals
# =========================================================
def extract_family_signals(text, family):
    lowered = normalize_space(text).lower()
    found = []
    for label, patterns in SIGNAL_PATTERNS[family].items():
        if any(re.search(pattern, lowered) for pattern in patterns):
            found.append(label)
    return unique_keep_order(found)


def extraction_insufficient_reason(row_dict, signal_family_count, signal_total_count):
    if not normalize_space(row_dict.get("target_review_text_for_extraction", "")):
        return "missing_target_review"

    query_safe_token_count = int(row_dict.get("query_safe_token_count_05", 0) or 0)
    direct_repair_applied = bool(row_dict.get("direct_cue_repair_applied", False))
    source_family_count = int(row_dict.get("query_safe_signal_family_count", 0) or 0)
    source_total_count = int(row_dict.get("query_safe_signal_total_count", 0) or 0)

    if query_safe_token_count < EXPECTED_MIN_QUERY_SAFE_TOKEN_COUNT:
        return "direct_cue_scrub_removed_required_evidence" if direct_repair_applied else "low_query_safe_token_count"
    if signal_family_count < EXPECTED_MIN_QUERY_SAFE_SIGNAL_FAMILIES:
        if direct_repair_applied and source_family_count >= EXPECTED_MIN_QUERY_SAFE_SIGNAL_FAMILIES:
            return "direct_cue_scrub_removed_required_evidence"
        return "low_signal_family_count"
    if signal_total_count < EXPECTED_MIN_QUERY_SAFE_SIGNAL_TOTAL:
        if direct_repair_applied and source_total_count >= EXPECTED_MIN_QUERY_SAFE_SIGNAL_TOTAL:
            return "direct_cue_scrub_removed_required_evidence"
        return "low_signal_total_count"
    return "sufficient"


signal_rows = []
for row in source.itertuples(index=False):
    row_dict = row._asdict()
    safe_text = row_dict.get("query_safe_review_text", "")
    signals = {family: extract_family_signals(safe_text, family) for family in QUERY_SAFE_SIGNAL_FAMILIES}
    signal_family_count = int(sum(1 for values in signals.values() if values))
    signal_total_count = int(sum(len(values) for values in signals.values()))
    insufficient_reason = extraction_insufficient_reason(row_dict, signal_family_count, signal_total_count)
    extraction_sufficient = insufficient_reason == "sufficient"

    signal_rows.append({
        "case_id": str(row_dict.get("case_id", "")),
        "user_id": str(row_dict.get("user_id", "")),
        "target_parent_asin": str(row_dict.get("parent_asin", "")),
        "target_rank_desc": int(row_dict.get("target_rank_desc", 0)),
        "target_timestamp_ms": int(row_dict.get("target_timestamp_ms", row_dict.get("review_timestamp_ms", 0))),
        "target_review_datetime": normalize_space(row_dict.get("review_datetime", "")),
        "target_selection_mode": str(row_dict.get("target_selection_mode", EXPECTED_TARGET_SELECTION_MODE)),
        "regime": str(row_dict.get("regime", "")),
        "sampling_bracket": str(row_dict.get("sampling_bracket", "")),
        "initial_selected": bool(row_dict.get("initial_selected", False)),
        "signal_source": SIGNAL_SOURCE,
        "query_evidence_source": QUERY_EVIDENCE_SOURCE,
        "item_metadata_evidence_used": False,
        "historical_review_evidence_used": False,
        "user_prior_evidence_used": False,
        "rating_evidence_used": False,
        "sentiment_evidence_used": False,
        "target_metadata_fallback_used": False,
        "item_context_fallback_used": False,
        "intended_use": INTENDED_USE,
        "query_safe_residual_text": normalize_space(safe_text),
        "qs_concern_signals": pipe_join(signals["concern"]),
        "qs_skin_type_signals": pipe_join(signals["skin_type"]),
        "qs_benefit_signals": pipe_join(signals["benefit"]),
        "qs_ingredient_signals": pipe_join(signals["ingredient"]),
        "qs_product_type_or_form_texture_signals": pipe_join(signals["product_type_or_form_texture"]),
        "qs_usage_context_signals": pipe_join(signals["usage_context"]),
        "query_safe_signal_family_count": signal_family_count,
        "query_safe_signal_total_count": signal_total_count,
        "signal_family_count": signal_family_count,
        "signal_total_count": signal_total_count,
        "target_review_token_count": int(
            row_dict.get("target_review_token_count", row_dict.get("target_review_token_count_05", 0))
        ),
        "query_safe_token_count": int(row_dict.get("query_safe_token_count_05", 0)),
        "review_signal_extraction_sufficient": bool(extraction_sufficient),
        "review_only_query_evidence_sufficient": bool(extraction_sufficient),
        "extraction_insufficient_reason": insufficient_reason,
        "direct_cue_repair_applied": bool(row_dict.get("direct_cue_repair_applied", False)),
        "direct_cue_warning_count": int(row_dict.get("direct_cue_warning_count", 0) or 0),
    })

review_signal_df = pd.DataFrame(signal_rows)
review_signal_df = review_signal_df.sort_values(["regime", "sampling_bracket", "case_id"]).reset_index(drop=True)

review_signal_output_row_count = int(len(review_signal_df))
review_signal_sufficient_count = int(review_signal_df["review_signal_extraction_sufficient"].sum())
review_signal_insufficient_count = int(review_signal_output_row_count - review_signal_sufficient_count)


In [10]:
# =========================================================
# Common Review Signal Role Annotation
# =========================================================
_review_signal_frame_name = "review_signal_df"
_review_signal_frame = globals().get(_review_signal_frame_name)
if _review_signal_frame is None:
    raise RuntimeError(f"{_review_signal_frame_name} must exist before common review signal annotation.")

if "common_framework_version" not in _review_signal_frame.columns:
    _review_signal_frame["common_framework_version"] = COMMON_REVIEW_SIGNAL_FRAMEWORK_VERSION
if "review_signal_source_scope" not in _review_signal_frame.columns:
    _review_signal_frame["review_signal_source_scope"] = "target_review_query_safe_text"
if "is_downstream_query_safe" not in _review_signal_frame.columns:
    _review_signal_frame["is_downstream_query_safe"] = 1
if "common_signal_family_count" not in _review_signal_frame.columns:
    source_col = "query_safe_signal_family_count" if "query_safe_signal_family_count" in _review_signal_frame.columns else "signal_family_count"
    _review_signal_frame["common_signal_family_count"] = pd.to_numeric(_review_signal_frame[source_col], errors="coerce").fillna(0).astype(int)
if "common_signal_total_count" not in _review_signal_frame.columns:
    source_col = "query_safe_signal_total_count" if "query_safe_signal_total_count" in _review_signal_frame.columns else "signal_total_count"
    _review_signal_frame["common_signal_total_count"] = pd.to_numeric(_review_signal_frame[source_col], errors="coerce").fillna(0).astype(int)


def _exact_phrase_hits(text, phrases):
    normalized = normalize_space(text).lower()
    hits = []
    for phrase in phrases:
        if re.search(r"(?<![a-z0-9])" + re.escape(phrase.lower()) + r"(?![a-z0-9])", normalized):
            hits.append(phrase)
    return sorted(set(hits))


def _pattern_phrase_hits(text, phrase_patterns):
    normalized = normalize_space(text).lower()
    return sorted({
        label
        for label, pattern in phrase_patterns.items()
        if re.search(pattern, normalized, flags=re.IGNORECASE)
    })


def _pipe_union_from_columns(row, columns):
    values = []
    seen = set()
    for col in columns:
        for part in str(row.get(col, "") or "").split(" | "):
            part = normalize_space(part)
            key = part.lower()
            if key and key not in seen:
                seen.add(key)
                values.append(part)
    return " | ".join(values)


if "source" not in globals() or "query_safe_review_text" not in source.columns:
    raise RuntimeError(
        "Query-audit annotation requires source.query_safe_review_text."
    )

_face_review_audit_df = source[["case_id", "query_safe_review_text"]].copy()
_face_review_audit_df["common_category_anchor_terms"] = _face_review_audit_df["query_safe_review_text"].map(
    lambda text: " | ".join(_exact_phrase_hits(text, FACE_REVIEW_CATEGORY_ANCHORS))
)
_face_review_audit_df["common_generic_utility_terms"] = _face_review_audit_df["query_safe_review_text"].map(
    lambda text: " | ".join(_exact_phrase_hits(text, FACE_REVIEW_GENERIC_UTILITY_TOKENS))
)
_face_review_audit_df["common_context_dependent_utility_terms"] = _face_review_audit_df["query_safe_review_text"].map(
    lambda text: " | ".join(_exact_phrase_hits(text, FACE_REVIEW_CONTEXT_DEPENDENT_TOKENS))
)
_face_review_audit_df["common_specific_support_phrases"] = _face_review_audit_df["query_safe_review_text"].map(
    lambda text: " | ".join(_pattern_phrase_hits(text, FACE_REVIEW_SPECIFIC_SUPPORT_PATTERNS))
)
_face_review_audit_df["common_generic_support_warning"] = _face_review_audit_df.apply(
    lambda row: int(
        bool(re.search(r"\bsupports?\b", normalize_space(row["query_safe_review_text"]).lower()))
        and not bool(normalize_space(row["common_specific_support_phrases"]))
    ),
    axis=1,
)
_face_review_audit_df = _face_review_audit_df.drop(columns=["query_safe_review_text"])

_review_signal_frame = _review_signal_frame.merge(
    _face_review_audit_df,
    on="case_id",
    how="left",
    validate="one_to_one",
)

_face_signal_columns = [
    "qs_concern_signals",
    "qs_skin_type_signals",
    "qs_benefit_signals",
    "qs_ingredient_signals",
    "qs_product_type_or_form_texture_signals",
    "qs_usage_context_signals",
]
_review_signal_frame["common_specific_signal_seed_text"] = _review_signal_frame.apply(
    lambda row: _pipe_union_from_columns(row, _face_signal_columns),
    axis=1,
)
_review_signal_frame["query_safe_residual_text"] = (
    _review_signal_frame["query_safe_residual_text"]
    .fillna("")
    .astype(str)
    .map(normalize_space)
)
_review_signal_frame["common_possible_entity_fragment_signals"] = ""
_review_signal_frame["common_query_audit_policy_version"] = "query_c_audit_v1"

required_common_audit_cols = [
    "common_category_anchor_terms",
    "common_generic_utility_terms",
    "common_context_dependent_utility_terms",
    "common_specific_support_phrases",
    "common_generic_support_warning",
    "common_specific_signal_seed_text",
    "common_possible_entity_fragment_signals",
    "common_query_audit_policy_version",
]
missing_common_audit_cols = [col for col in required_common_audit_cols if col not in _review_signal_frame.columns]
if missing_common_audit_cols:
    raise RuntimeError(f"Missing common query-audit columns: {missing_common_audit_cols}")

common_family_rows = []
for family, role in COMMON_REVIEW_SIGNAL_FAMILY_TO_ROLE.items():
    candidate_cols = [
        "qs_" + family + "_signals",
        family + "_signals",
        "has_" + family,
    ]
    present_cols = [col for col in candidate_cols if col in _review_signal_frame.columns]
    for col in present_cols:
        series = _review_signal_frame[col]
        if col.startswith("has_"):
            non_empty = pd.to_numeric(series, errors="coerce").fillna(0).astype(int).gt(0)
            unique_value_count = int(series.nunique(dropna=True))
        else:
            non_empty = series.fillna("").astype(str).str.strip().ne("")
            unique_values = set()
            for value in series.fillna("").astype(str):
                unique_values.update([part.strip().lower() for part in value.split(" | ") if part.strip()])
            unique_value_count = len(unique_values)
        common_family_rows.append({
            "signal_family": family,
            "common_role": role,
            "source_column": col,
            "non_empty_rows": int(non_empty.sum()),
            "coverage_rate": float(non_empty.mean()) if len(non_empty) else 0.0,
            "unique_value_count": int(unique_value_count),
        })

common_review_signal_family_coverage_df = pd.DataFrame(common_family_rows)
globals()[_review_signal_frame_name] = _review_signal_frame

# Common coverage is written in the summary output cells.


In [11]:
# =========================================================
# Build Summaries and Save Downstream-Safe Signal Output
# =========================================================
signal_cols = [
    "qs_concern_signals",
    "qs_skin_type_signals",
    "qs_benefit_signals",
    "qs_ingredient_signals",
    "qs_product_type_or_form_texture_signals",
    "qs_usage_context_signals",
]

coverage_rows = []
for col in signal_cols:
    coverage_rows.append({
        "section": "signal_family_coverage",
        "signal_family": col.replace("qs_", "").replace("_signals", ""),
        "non_empty_rows": int(review_signal_df[col].fillna("").astype(str).str.strip().ne("").sum()),
        "coverage_ratio": float(review_signal_df[col].fillna("").astype(str).str.strip().ne("").mean()) if len(review_signal_df) else 0.0,
        "unique_signal_values": int(len(set(" | ".join(review_signal_df[col].fillna("").astype(str)).split(" | ")) - {""})),
    })
signal_coverage_df = pd.DataFrame(coverage_rows)

coverage_by_regime_df = (
    review_signal_df.groupby("regime", dropna=False)
    .agg(
        rows=("case_id", "count"),
        users=("user_id", "nunique"),
        items=("target_parent_asin", "nunique"),
        rows_with_any_signal=("query_safe_signal_total_count", lambda s: int((s > 0).sum())),
        review_signal_extraction_sufficient_cases=("review_signal_extraction_sufficient", "sum"),
        avg_signal_total_count=("query_safe_signal_total_count", "mean"),
        avg_signal_family_count=("query_safe_signal_family_count", "mean"),
    )
    .reset_index()
)
coverage_by_regime_df["section"] = "regime_coverage"
coverage_by_regime_df["signal_coverage_ratio"] = np.where(
    coverage_by_regime_df["rows"] > 0,
    coverage_by_regime_df["rows_with_any_signal"] / coverage_by_regime_df["rows"],
    0.0,
)

family_top_rows = []
for col in signal_cols:
    counter = Counter()
    for value in review_signal_df[col].fillna("").astype(str):
        counter.update([part.strip() for part in value.split(" | ") if part.strip()])
    for signal, count in counter.most_common(20):
        family_top_rows.append({
            "section": "top_signal_counts",
            "signal_family": col.replace("qs_", "").replace("_signals", ""),
            "signal": signal,
            "count": int(count),
        })
top_signals_df = pd.DataFrame(family_top_rows, columns=["section", "signal_family", "signal", "count"])

supply_rows = []
for regime in EXPECTED_REGIMES:
    regime_df = review_signal_df.loc[review_signal_df["regime"].eq(regime)].copy()
    initial_mask = regime_df["initial_selected"].astype(bool)
    reserve_mask = ~initial_mask
    sufficient_mask = regime_df["review_signal_extraction_sufficient"].astype(bool)
    eligible_cases = int(regime_df["case_id"].nunique())
    initial_selected_cases = int(regime_df.loc[initial_mask, "case_id"].nunique())
    reserve_cases = int(regime_df.loc[reserve_mask, "case_id"].nunique())
    sufficient_cases = int(regime_df.loc[sufficient_mask, "case_id"].nunique())
    insufficient_cases = int(eligible_cases - sufficient_cases)
    sufficient_initial_cases = int(regime_df.loc[initial_mask & sufficient_mask, "case_id"].nunique())
    insufficient_initial_cases = int(initial_selected_cases - sufficient_initial_cases)
    sufficient_reserve_cases = int(regime_df.loc[reserve_mask & sufficient_mask, "case_id"].nunique())
    insufficient_reserve_cases = int(reserve_cases - sufficient_reserve_cases)
    supply_rows.append({
        "regime": regime,
        "eligible_cases": eligible_cases,
        "initial_selected_cases": initial_selected_cases,
        "reserve_cases": reserve_cases,
        "review_signal_extraction_sufficient_cases": sufficient_cases,
        "review_signal_extraction_insufficient_cases": insufficient_cases,
        "sufficient_initial_cases": sufficient_initial_cases,
        "insufficient_initial_cases": insufficient_initial_cases,
        "sufficient_reserve_cases": sufficient_reserve_cases,
        "insufficient_reserve_cases": insufficient_reserve_cases,
    })
regime_supply_summary_df = pd.DataFrame(supply_rows)

for row in regime_supply_summary_df.to_dict(orient="records"):
    if row["eligible_cases"] != row["initial_selected_cases"] + row["reserve_cases"]:
        raise RuntimeError(f"Initial/reserve partition failed for regime {row['regime']}.")
    if row["eligible_cases"] != row["review_signal_extraction_sufficient_cases"] + row["review_signal_extraction_insufficient_cases"]:
        raise RuntimeError(f"Sufficient/insufficient partition failed for regime {row['regime']}.")
    if row["reserve_cases"] != row["sufficient_reserve_cases"] + row["insufficient_reserve_cases"]:
        raise RuntimeError(f"Reserve sufficient/insufficient partition failed for regime {row['regime']}.")

insufficient_audit_cols = [
    "case_id",
    "user_id",
    "target_parent_asin",
    "target_timestamp_ms",
    "target_review_datetime",
    "regime",
    "sampling_bracket",
    "initial_selected",
    "review_signal_extraction_sufficient",
    "extraction_insufficient_reason",
    "query_safe_token_count",
    "query_safe_signal_family_count",
    "query_safe_signal_total_count",
    "common_specific_signal_seed_text",
    "direct_cue_repair_applied",
    "direct_cue_warning_count",
]
insufficient_audit_df = review_signal_df.loc[
    ~review_signal_df["review_signal_extraction_sufficient"].astype(bool),
    insufficient_audit_cols,
].copy()

direct_cue_summary_df = pd.DataFrame([{
    "actual_direct_cue_removed_rows": actual_direct_cue_removed_rows,
    "normalization_only_changed_rows": normalization_only_changed_rows,
    "asin_removed_rows": asin_removed_rows,
    "seller_manufacturer_removed_rows": seller_manufacturer_removed_rows,
    "package_identifier_removed_rows": package_identifier_removed_rows,
    "metadata_phrase_removed_rows": metadata_phrase_removed_rows,
    "title_phrase_removed_rows": title_phrase_removed_rows,
    "metadata_token_removed_rows": metadata_token_removed_rows,
    "direct_brand_identifier_leakage_flagged_rows_after_repair": direct_leakage_flag_count,
    "package_or_identifier_warning_rows_after_repair": package_or_identifier_warning_count,
    "exact_title_phrase_warning_rows_after_repair": title_exact_leak_warning_count,
}])

summary_export_df = pd.concat([signal_coverage_df, coverage_by_regime_df, top_signals_df], ignore_index=True, sort=False)

REQUIRED_REVIEW_SIGNAL_OUTPUT_COLS = [
    "case_id",
    "user_id",
    "regime",
    "target_parent_asin",
    "target_timestamp_ms",
    "target_review_datetime",
    "initial_selected",
    "signal_source",
    "query_evidence_source",
    "item_metadata_evidence_used",
    "historical_review_evidence_used",
    "user_prior_evidence_used",
    "rating_evidence_used",
    "sentiment_evidence_used",
    "target_metadata_fallback_used",
    "item_context_fallback_used",
    "intended_use",
    "query_safe_residual_text",
    "qs_concern_signals",
    "qs_skin_type_signals",
    "qs_benefit_signals",
    "qs_ingredient_signals",
    "qs_product_type_or_form_texture_signals",
    "qs_usage_context_signals",
    "common_specific_support_phrases",
    "common_specific_signal_seed_text",
    "query_safe_signal_family_count",
    "query_safe_signal_total_count",
    "review_signal_extraction_sufficient",
    "review_only_query_evidence_sufficient",
    "extraction_insufficient_reason",
    "direct_cue_repair_applied",
    "direct_cue_warning_count",
    "target_review_token_count",
    "query_safe_token_count",
]
missing_review_signal_output_cols = [col for col in REQUIRED_REVIEW_SIGNAL_OUTPUT_COLS if col not in review_signal_df.columns]
if missing_review_signal_output_cols:
    raise RuntimeError(f"Missing required Notebook 05 output columns: {missing_review_signal_output_cols}")

ALLOWED_REVIEW_SIGNAL_POLICY_COLS = {
    "item_metadata_evidence_used",
    "historical_review_evidence_used",
    "user_prior_evidence_used",
    "rating_evidence_used",
    "sentiment_evidence_used",
    "target_metadata_fallback_used",
    "item_context_fallback_used",
}
POLICY_FALSE_COLUMNS = sorted(ALLOWED_REVIEW_SIGNAL_POLICY_COLS)
FORBIDDEN_REVIEW_SIGNAL_OUTPUT_EXACT_COLS = {
    "parent_asin",
    "asin",
    "target_asin",
    "item_asin",
    "title",
    "target_review_title",
    "review_title",
    "item_title",
    "target_review_text",
    "heldout_review_text",
    "review_text",
    "review_body",
    "raw_review_text",
    "target_review_body",
    "brand",
    "brand_meta",
    "manufacturer",
    "seller",
    "product_line",
    "rating",
    "sentiment",
    "helpful_vote",
    "total_vote",
    "verified_purchase",
    "prompt",
    "response",
    "llm_response",
    "query_evidence",
    "canonical_retrieval_text",
    "canonical_metadata_text",
    "profile_source_text_dedup_seed",
    "review_reputation_facet_text",
    "historical_review_reputation_text",
    "query_safe_facet_text",
    "functional_facet_text",
}

for col in review_signal_df.columns:
    lower_col = str(col).lower()
    if lower_col in ALLOWED_REVIEW_SIGNAL_POLICY_COLS:
        continue
    if (
        lower_col in FORBIDDEN_REVIEW_SIGNAL_OUTPUT_EXACT_COLS
        or lower_col.startswith("review_reputation_")
        or lower_col.startswith("historical_review_")
        or lower_col.startswith("user_prior_")
        or lower_col.startswith("itemctx_")
    ):
        raise RuntimeError(f"Forbidden column found in Notebook 05 signal output: {col}")

for col in POLICY_FALSE_COLUMNS:
    if review_signal_df[col].dtype != bool:
        review_signal_df[col] = review_signal_df[col].astype(bool)
    if review_signal_df[col].any():
        raise RuntimeError(f"{col} must be False for every output row.")

if not review_signal_df["review_signal_extraction_sufficient"].eq(review_signal_df["review_only_query_evidence_sufficient"]).all():
    raise RuntimeError("Notebook 05 sufficiency aliases disagree.")

review_signal_df.to_parquet(REVIEW_SIGNAL_PATH, index=False)
review_signal_df.to_csv(REVIEW_SIGNAL_CSV_PATH, index=False, encoding="utf-8-sig")
summary_export_df.to_csv(OUTPUT_DIR / "face_review_signal_summary.csv", index=False, encoding="utf-8-sig")
summary_export_df.to_json(OUTPUT_DIR / "face_review_signal_summary.json", orient="records", force_ascii=False, indent=2)
signal_coverage_df.to_csv(OUTPUT_DIR / "face_review_signal_family_coverage.csv", index=False, encoding="utf-8-sig")
regime_supply_summary_df.to_csv(OUTPUT_DIR / "face_review_signal_regime_supply_summary.csv", index=False, encoding="utf-8-sig")
insufficient_audit_df.to_csv(OUTPUT_DIR / "face_review_signal_extraction_insufficient_audit.csv", index=False, encoding="utf-8-sig")
direct_cue_summary_df.to_csv(OUTPUT_DIR / "face_review_signal_direct_cue_summary.csv", index=False, encoding="utf-8-sig")


In [12]:
# =========================================================
# Final Validation, Diagnostics, and Manifest
# =========================================================
if len(review_signal_df) != EXPECTED_OUTPUT_ROWS:
    raise RuntimeError(f"face_review_signals.parquet must have {EXPECTED_OUTPUT_ROWS} rows, found {len(review_signal_df)}")
if review_signal_df["case_id"].duplicated().any():
    raise RuntimeError("case_id must be unique in face_review_signals.parquet.")
if review_signal_df["case_id"].isna().any() or review_signal_df["case_id"].astype(str).str.strip().eq("").any():
    raise RuntimeError("case_id must be non-empty in face_review_signals.parquet.")
if review_signal_df["user_id"].duplicated().any():
    raise RuntimeError("user_id must be unique in face_review_signals.parquet.")
if review_signal_df["user_id"].isna().any() or review_signal_df["user_id"].astype(str).str.strip().eq("").any():
    raise RuntimeError("user_id must be non-empty in face_review_signals.parquet.")

output_case_ids = set(review_signal_df["case_id"].astype(str))
input_case_ids = set(eligible_cases_df["case_id"].astype(str))
if output_case_ids != input_case_ids:
    raise RuntimeError("Notebook 05 output case_id set must equal the full eligible-pool case_id set.")

output_regime_counts = review_signal_df["regime"].value_counts().reindex(EXPECTED_REGIMES, fill_value=0).astype(int)
observed_output_regime_counts = {regime: int(output_regime_counts.get(regime, 0)) for regime in EXPECTED_REGIMES}
if observed_output_regime_counts != EXPECTED_ELIGIBLE_REGIME_COUNTS:
    raise RuntimeError(
        f"Output regime counts mismatch. Actual: {observed_output_regime_counts}. "
        f"Expected: {EXPECTED_ELIGIBLE_REGIME_COUNTS}."
    )
if not review_signal_df["signal_source"].eq(SIGNAL_SOURCE).all():
    raise RuntimeError("signal_source must be heldout_target_review for every output row.")
if not review_signal_df["intended_use"].eq(INTENDED_USE).all():
    raise RuntimeError("intended_use must be synthetic_query_generation_only for every output row.")
if not review_signal_df["query_evidence_source"].eq(QUERY_EVIDENCE_SOURCE).all():
    raise RuntimeError("query_evidence_source must be target_review_only.")
if not review_signal_df["target_selection_mode"].eq(EXPECTED_TARGET_SELECTION_MODE).all():
    raise RuntimeError("Notebook 05 output target_selection_mode mismatch.")
if not review_signal_df["target_rank_desc"].between(1, MAX_TARGET_RANK_ALLOWED).all():
    raise RuntimeError(f"Notebook 05 output target_rank_desc must be between 1 and {MAX_TARGET_RANK_ALLOWED}.")

for col in POLICY_FALSE_COLUMNS:
    if review_signal_df[col].dtype != bool:
        raise RuntimeError(f"{col} must have boolean dtype.")
    if review_signal_df[col].any():
        raise RuntimeError(f"{col} must be False for every output row.")

for col in review_signal_df.columns:
    lower_col = str(col).lower()
    if lower_col in ALLOWED_REVIEW_SIGNAL_POLICY_COLS:
        continue
    if (
        lower_col in FORBIDDEN_REVIEW_SIGNAL_OUTPUT_EXACT_COLS
        or lower_col.startswith("review_reputation_")
        or lower_col.startswith("historical_review_")
        or lower_col.startswith("user_prior_")
        or lower_col.startswith("itemctx_")
    ):
        raise RuntimeError(f"Forbidden column found in Notebook 05 signal output: {col}")

if metadata_matched_rows != EXPECTED_OUTPUT_ROWS:
    raise RuntimeError("Final validation failed: metadata_matched_rows must equal EXPECTED_OUTPUT_ROWS.")
if not source["item_metadata_exists"].all():
    raise RuntimeError("Final validation failed: item_metadata_exists must be True for all eligible rows.")
if itemctx_title_non_empty != EXPECTED_OUTPUT_ROWS:
    raise RuntimeError("Final validation failed: itemctx_title must be non-empty for all eligible rows.")
if direct_leakage_flag_count != 0:
    raise RuntimeError("Final validation failed: direct brand/identifier leakage remains after repair.")
if regime_supply_summary_df["eligible_cases"].sum() != EXPECTED_OUTPUT_ROWS:
    raise RuntimeError("Regime supply summary does not cover the full eligible pool.")
if regime_supply_summary_df["initial_selected_cases"].sum() != EXPECTED_INITIAL_SAMPLE_ROWS:
    raise RuntimeError("Initial selected cases do not match the Notebook 03 initial sample size.")
if regime_supply_summary_df["reserve_cases"].sum() != EXPECTED_OUTPUT_ROWS - EXPECTED_INITIAL_SAMPLE_ROWS:
    raise RuntimeError("Reserve cases do not partition the eligible pool.")

created_outputs = [
    REVIEW_SIGNAL_PATH,
    REVIEW_SIGNAL_CSV_PATH,
    OUTPUT_DIR / "face_review_signal_summary.csv",
    OUTPUT_DIR / "face_review_signal_summary.json",
    OUTPUT_DIR / "face_review_signal_family_coverage.csv",
    OUTPUT_DIR / "face_review_signal_regime_supply_summary.csv",
    OUTPUT_DIR / "face_review_signal_extraction_insufficient_audit.csv",
    OUTPUT_DIR / "face_review_signal_direct_cue_summary.csv",
    OUTPUT_DIR / "face_review_signal_diagnostics.json",
    REVIEW_SIGNAL_MANIFEST_PATH,
]

direct_cue_audit_summary = {
    "direct_cue_removal_used_item_metadata": True,
    "direct_cue_removal_policy": "remove_asin_seller_brand_identifier_package_and_exact_title_cues__harmonized_facet_brand_sources",
    "actual_direct_cue_removed_rows": actual_direct_cue_removed_rows,
    "normalization_only_changed_rows": normalization_only_changed_rows,
    "asin_removed_rows": asin_removed_rows,
    "seller_manufacturer_removed_rows": seller_manufacturer_removed_rows,
    "package_identifier_removed_rows": package_identifier_removed_rows,
    "metadata_phrase_removed_rows": metadata_phrase_removed_rows,
    "title_phrase_removed_rows": title_phrase_removed_rows,
    "metadata_token_removed_rows": metadata_token_removed_rows,
    "direct_cue_repaired_rows": actual_direct_cue_removed_rows,
    "direct_brand_identifier_leakage_flagged_rows_after_repair": direct_leakage_flag_count,
    "package_or_identifier_warning_rows_after_repair": package_or_identifier_warning_count,
    "exact_title_phrase_warning_rows_after_repair": title_exact_leak_warning_count,
}
sufficient_by_regime = {
    regime: int(review_signal_df.loc[review_signal_df["regime"].eq(regime), "review_signal_extraction_sufficient"].sum())
    for regime in EXPECTED_REGIMES
}
insufficient_by_regime = {
    regime: int((~review_signal_df.loc[review_signal_df["regime"].eq(regime), "review_signal_extraction_sufficient"].astype(bool)).sum())
    for regime in EXPECTED_REGIMES
}
sufficient_reserve_by_regime = {
    row["regime"]: int(row["sufficient_reserve_cases"])
    for row in regime_supply_summary_df.to_dict(orient="records")
}
input_summary = {
    "eligible_pool_rows": int(len(eligible_cases_df)),
    "eligible_pool_regime_counts": observed_output_regime_counts,
    "initial_selected_rows": int(initial_selected_row_count),
    "reserve_rows": int(reserve_row_count),
    "initial_selected_regime_counts": initial_regime_counts,
}

item_schema_summary = {
    "item_schema_path": str(ITEM_SCHEMA_PATH),
    "required_item_schema_columns": REQUIRED_ITEM_SCHEMA_COLS,
    "harmonized_required_item_schema_columns": HARMONIZED_REQUIRED_ITEM_SCHEMA_COLS,
    "item_schema_rows": int(len(item_schema)),
    "facet_policy_version": HARMONIZED_FACET_POLICY_VERSION,
    "brand_policy": EXPECTED_BRAND_POLICY,
    "identifier_policy": EXPECTED_IDENTIFIER_POLICY,
    "review_reputation_columns_present_but_ignored": review_reputation_cols_present,
}

diagnostics_payload = {
    "stage": STAGE,
    "schema_version": REVIEW_SIGNAL_SCHEMA_VERSION,
    "category_id": CATEGORY_ID,
    "category_folder": CATEGORY_FOLDER,
    "category_label": CATEGORY_LABEL,
    "input_scope": "full_user_level_eligible_pool",
    "purpose": "heldout_target_review_signals_for_synthetic_query_generation_only",
    "input_summary": input_summary,
    "output_signal_rows": int(len(review_signal_df)),
    "unique_users": int(review_signal_df["user_id"].nunique()),
    "evaluation_window_months": int(sample_manifest["evaluation_window_months"]),
    "target_selection_mode": EXPECTED_TARGET_SELECTION_MODE,
    "max_target_rank_allowed": MAX_TARGET_RANK_ALLOWED,
    "target_rank_convention": sample_manifest["target_rank_convention"],
    "regime_counts": observed_output_regime_counts,
    "signal_source": SIGNAL_SOURCE,
    "query_evidence_source": QUERY_EVIDENCE_SOURCE,
    "intended_use": INTENDED_USE,
    "review_signal_extraction_sufficient_cases": int(review_signal_df["review_signal_extraction_sufficient"].sum()),
    "review_signal_extraction_insufficient_cases": int((~review_signal_df["review_signal_extraction_sufficient"].astype(bool)).sum()),
    "review_signal_extraction_sufficient_by_regime": sufficient_by_regime,
    "review_signal_extraction_insufficient_by_regime": insufficient_by_regime,
    "sufficient_reserve_capacity_by_regime": sufficient_reserve_by_regime,
    "regime_supply_summary": json_safe_records(regime_supply_summary_df),
    "insufficient_reason_counts": review_signal_df["extraction_insufficient_reason"].value_counts().astype(int).to_dict(),
    "prior_profile_features_constructed": False,
    "prior_history_artifact_read": False,
    "item_metadata_evidence_used": False,
    "historical_review_evidence_used": False,
    "user_prior_evidence_used": False,
    "target_metadata_fallback_used": False,
    "item_context_fallback_used": False,
    "rating_evidence_used": False,
    "sentiment_evidence_used": False,
    "metadata_used_for_query_safety_filtering_only": True,
    "metadata_matched_rows": metadata_matched_rows,
    "metadata_match_rate": metadata_match_rate,
    **direct_cue_audit_summary,
    "item_review_reputation_used": False,
    "item_review_reputation_columns_present_but_ignored": review_reputation_cols_present,
    "item_schema_summary": item_schema_summary,
    "sentiment_like_families_removed_or_qc_only": SENTIMENT_LIKE_FAMILIES_REMOVED,
    "raw_review_text_exported": False,
    "raw_review_title_exported": False,
    "rating_used": False,
    "sentiment_used": False,
    "llm_used": False,
    "signal_family_coverage": json_safe_records(signal_coverage_df),
    "signal_coverage_by_regime": json_safe_records(coverage_by_regime_df),
    "top_signal_counts_by_family": json_safe_records(top_signals_df.groupby("signal_family", group_keys=False).head(10)) if len(top_signals_df) else [],
    "saved_file_paths": [str(path) for path in created_outputs],
}
with open(OUTPUT_DIR / "face_review_signal_diagnostics.json", "w", encoding="utf-8") as f:
    json.dump(diagnostics_payload, f, ensure_ascii=False, indent=2)

manifest = {
    "created_at_utc": pd.Timestamp.now(tz="UTC").isoformat(),
    "stage": STAGE,
    "schema_version": REVIEW_SIGNAL_SCHEMA_VERSION,
    "category_id": CATEGORY_ID,
    "category_folder": CATEGORY_FOLDER,
    "category_label": CATEGORY_LABEL,
    "input_scope": "full_user_level_eligible_pool",
    "project_root": str(PROJECT_ROOT),
    "input_paths": {name: str(path) for name, path in REQUIRED_INPUT_PATHS.items()},
    "output_paths": [str(path) for path in created_outputs],
    "input_eligible_pool_rows": EXPECTED_OUTPUT_ROWS,
    "output_signal_rows": EXPECTED_OUTPUT_ROWS,
    "eligible_pool_regime_counts": observed_output_regime_counts,
    "initial_selected_rows": int(initial_selected_row_count),
    "reserve_rows": int(reserve_row_count),
    "regime_supply_summary": json_safe_records(regime_supply_summary_df),
    "evaluation_window_months": int(sample_manifest["evaluation_window_months"]),
    "target_selection_mode": EXPECTED_TARGET_SELECTION_MODE,
    "max_target_rank_allowed": MAX_TARGET_RANK_ALLOWED,
    "target_rank_convention": sample_manifest["target_rank_convention"],
    "signal_source": SIGNAL_SOURCE,
    "query_evidence_source": QUERY_EVIDENCE_SOURCE,
    "intended_use": INTENDED_USE,
    "review_signal_extraction_sufficient_cases": int(review_signal_df["review_signal_extraction_sufficient"].sum()),
    "review_signal_extraction_insufficient_cases": int((~review_signal_df["review_signal_extraction_sufficient"].astype(bool)).sum()),
    "review_signal_extraction_sufficient_by_regime": sufficient_by_regime,
    "review_signal_extraction_insufficient_by_regime": insufficient_by_regime,
    "sufficient_reserve_capacity_by_regime": sufficient_reserve_by_regime,
    "prior_profile_features_constructed": False,
    "prior_history_artifact_read": False,
    "item_metadata_evidence_used": False,
    "historical_review_evidence_used": False,
    "user_prior_evidence_used": False,
    "target_metadata_fallback_used": False,
    "item_context_fallback_used": False,
    "rating_used": False,
    "sentiment_used": False,
    "llm_used": False,
    "raw_review_text_exported": False,
    "deterministic_rule_based_extraction_only": True,
}
with open(REVIEW_SIGNAL_MANIFEST_PATH, "w", encoding="utf-8") as f:
    json.dump(manifest, f, ensure_ascii=False, indent=2)

missing_outputs = [str(path) for path in created_outputs if not path.exists()]
if missing_outputs:
    raise RuntimeError(f"Expected Notebook 05 outputs were not written: {missing_outputs}")

validation_status = "PASS"


In [13]:
# =========================================================
# Final Summary
# =========================================================
print("Input row count:", eligible_input_row_count)
print("Output row count:", len(review_signal_df))
print("Sufficient count:", review_signal_sufficient_count)
print("Insufficient count:", review_signal_insufficient_count)
print("Validation status:", validation_status)


Input row count: 28348
Output row count: 28348
Sufficient count: 28156
Insufficient count: 192
Validation status: PASS


In [14]:
# =========================================================
# Common Review Signal Contract Export
# =========================================================
common_review_signal_contract = dict(COMMON_REVIEW_SIGNAL_CONTRACT)
_review_signal_frame = globals().get("review_signal_df")
common_review_signal_contract["input_scope"] = "full_user_level_eligible_pool"
common_review_signal_contract["output_paths"] = {
    "common_review_signal_contract": str(COMMON_REVIEW_SIGNAL_CONTRACT_PATH),
    "common_review_signal_family_coverage": str(COMMON_REVIEW_SIGNAL_FAMILY_COVERAGE_PATH),
}
common_review_signal_contract["row_count"] = int(len(_review_signal_frame)) if _review_signal_frame is not None else None
common_review_signal_contract["columns"] = list(_review_signal_frame.columns) if _review_signal_frame is not None else []
common_review_signal_contract["query_evidence_contract"] = {
    "query_evidence_source": QUERY_EVIDENCE_SOURCE,
    "item_metadata_evidence_used": False,
    "historical_review_evidence_used": False,
    "user_prior_evidence_used": False,
    "rating_evidence_used": False,
    "sentiment_evidence_used": False,
    "target_metadata_fallback_used": False,
    "item_context_fallback_used": False,
    "query_safe_residual_column": "query_safe_residual_text",
    "specific_signal_seed_column": "common_specific_signal_seed_text",
}
common_review_signal_contract["query_audit_fields"] = [
    "common_category_anchor_terms",
    "common_generic_utility_terms",
    "common_context_dependent_utility_terms",
    "common_specific_support_phrases",
    "common_generic_support_warning",
    "common_specific_signal_seed_text",
    "common_possible_entity_fragment_signals",
]
common_review_signal_contract["downstream_note"] = (
    "Existing qs_* signal columns are preserved. Notebook 06 uses full-pool "
    "target-review-only signals and remains responsible for final query-candidate selection."
)

COMMON_REVIEW_SIGNAL_CONTRACT_PATH.parent.mkdir(parents=True, exist_ok=True)
with open(COMMON_REVIEW_SIGNAL_CONTRACT_PATH, "w", encoding="utf-8") as f:
    json.dump(common_review_signal_contract, f, ensure_ascii=False, indent=2)

if "common_review_signal_family_coverage_df" in globals() and not common_review_signal_family_coverage_df.empty:
    common_review_signal_family_coverage_df.to_csv(COMMON_REVIEW_SIGNAL_FAMILY_COVERAGE_PATH, index=False, encoding="utf-8-sig")
